# 05 — Measuring thermal conductivity

Fourier's law says a heat flux $J$ through a material sustains a
temperature gradient: $J = -\kappa\, \nabla T$. Here we measure the
thermal conductivity $\kappa$ of a 2D Lennard-Jones liquid with the
**Müller-Plathe** trick (`fix thermal/conductivity`): periodically swap the
kinetic energy of the hottest atom in one slab with the coldest atom in
another. That imposes a *known* heat flux; the fluid answers with a
temperature gradient, which we read from a `fix ave/chunk` profile written
to a file — the same file-based workflow as
[basics/05](../basics/05-thermo-output-to-file.ipynb).

In [ ]:
%pip install lammps-js matplotlib

## Impose the flux

A wide, short box (aspect ratio 4:1). The swap region is slab 0 (cold,
left edge) and slab 10 of 20 (hot, middle), so heat flows through the two
halves in opposite directions. `f_mp` accumulates the total kinetic energy
moved by the swaps. At the end, `unfix tp` closes the profile file (an
averaging fix keeps its output file open while it is active), and we grab
the numbers the analysis needs before closing the session:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps

lmp = await lammps(output=None)
lmp.commands_string("""
units         lj
dimension     2
lattice       hex 0.7
region        box block 0 40 0 10 -0.1 0.1
create_box    1 box
create_atoms  1 box
mass          1 1.0
velocity      all create 0.9 87287 dist gaussian
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
fix           1 all nve
fix           2d all enforce2d
thermo        2000
run           4000
reset_timestep 0

compute       ck all chunk/atom bin/1d x lower 0.05 units reduced
fix           tp all ave/chunk 10 1000 20000 ck temp file tprofile.txt
fix           mp all thermal/conductivity 20 x 20
variable      swapped equal f_mp
run           20000
""")
e_swapped = lmp.extract_variable("swapped")
lmp.command("unfix tp")   # closes tprofile.txt so it lands in the notebook filesystem
t_run = 20000 * lmp.extract_global("dt")
box = lmp.extract_box()
Lx = box[1][0] - box[0][0]
Ly = box[1][1] - box[0][1]
lmp.close()
print(f"kinetic energy transported by swaps: {e_swapped:.1f} (LJ units) "
      f"over t = {t_run:.0f}")

## Read the temperature profile and fit the gradient

`ave/chunk` wrote one block of 20 bins to `tprofile.txt` (check the file
browser — same file-based workflow as
[basics/05](../basics/05-thermo-output-to-file.ipynb)), time-averaged over
the second half of the run; the first half is the transient while the
gradient builds up. The profile is tent-shaped — cold at the swap slab on
the left edge, hot in the middle — so we fit a straight line to each half
and average the slopes:

In [ ]:
raw = np.loadtxt("tprofile.txt", skiprows=4)     # bin, coord, natoms, temp
xfrac, temp = raw[:, 1], raw[:, 3]
x = xfrac * Lx

up, down = slice(1, 10), slice(11, 20)           # skip the swap bins
s1 = np.polyfit(x[up], temp[up], 1)
s2 = np.polyfit(x[down], temp[down], 1)
dTdx = (abs(s1[0]) + abs(s2[0])) / 2

# Fourier: J = kappa dT/dx. The swapped energy splits over two directions
# and crosses area Ly (a length, in 2D).
J = e_swapped / (2 * t_run * Ly)
kappa = J / dTdx
print(f"dT/dx = {dTdx:.4f}   J = {J:.4f}   →   κ ≈ {kappa:.2f} (LJ units)")

plt.figure(figsize=(6.5, 3.6))
plt.plot(x, temp, "o", ms=4)
plt.plot(x[up], np.polyval(s1, x[up]), "--", color="tab:red")
plt.plot(x[down], np.polyval(s2, x[down]), "--", color="tab:red")
plt.xlabel("x"); plt.ylabel("temperature")
plt.title(f"Steady-state gradient  →  κ ≈ {kappa:.2f}")
plt.tight_layout(); plt.show()

A real transport coefficient from a 30-second browser simulation. Things
worth trying: double the run length (the profile gets smoother), change the
density, or measure at several temperatures — for a dense LJ liquid κ
falls as it warms.

Next: [06 — Polymer chains](06-polymer-chains.ipynb), where numpy builds
the molecules.